# Serialización del mejor modelo y MLflow Model Registry

En este notebook serializamos el pipeline completo del mejor modelo
(**Exp 0: LogisticRegression + TF-IDF**, F1-macro test: 0.9530)
y lo registramos en el MLflow Model Registry.

Pasos:
1. Carga del mejor modelo
2. Serialización completa del pipeline con naming convention claro
3. Loguear el modelo en MLflow con `mlflow.sklearn.log_model`
4. Registrar en el Model Registry y transitar a stage `Production`
5. Verificación del registro

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Carga del mejor modelo

In [16]:
import json
import joblib

with open("model/mejor_modelo_seleccion.json", encoding="utf-8") as f:
    seleccion = json.load(f)

modelo = joblib.load(seleccion["model_file"])
tfidf  = joblib.load(seleccion["tfidf_file"])

print(f"Modelo seleccionado: {seleccion['nombre']}")
print(f"Tipo: {seleccion['model_type']}")
print(f"Vocabulario TF-IDF: {len(tfidf.vocabulary_)} términos")
if hasattr(modelo, "classes_"):
    print(f"Clases: {sorted(modelo.classes_.tolist())}")
print(f"\nMétricas en test:")
print(f"  F1-macro:  {seleccion['test_f1_macro']}")
print(f"  Accuracy:  {seleccion['test_accuracy']}")
print(f"  ROC AUC:   {seleccion['test_roc_auc']}")

Modelo seleccionado: Exp 0: LogReg + TF-IDF
Tipo: LogisticRegression
Vocabulario TF-IDF: 3773 términos
Clases: ['alto_riesgo', 'inaceptable', 'riesgo_limitado', 'riesgo_minimo']

Métricas en test:
  F1-macro:  0.9053
  Accuracy:  0.9111
  ROC AUC:   0.9948


## 2. Serialización completa del pipeline

In [17]:
from functions import guardar_pipeline_completo

features_label = "tfidf_bigramas" if not seleccion["needs_manual_features"] else "tfidf_bigramas+manuales"

path_modelo, path_tfidf, path_meta = guardar_pipeline_completo(
    modelo=modelo,
    tfidf=tfidf,
    label_encoder=None,
    metadata={
        "experimento":   seleccion["experimento"],
        "nombre":        seleccion["nombre"],
        "test_f1_macro": seleccion["test_f1_macro"],
        "test_accuracy": seleccion["test_accuracy"],
        "test_roc_auc":  seleccion["test_roc_auc"],
        "features":      features_label,
        "tfidf_n_terms": len(tfidf.vocabulary_),
        "criterio":      seleccion["criterio"],
    },
    output_dir="model",
)

Modelo guardado:      model\mejor_modelo.joblib
Vectorizador guardado: model\mejor_modelo_tfidf.joblib
Metadata guardada:    model\model_metadata.json


## 3. Loguear el modelo en MLflow con sklearn.log_model

Usamos `mlflow.sklearn.log_model` con `registered_model_name` para que
MLflow cree automáticamente la primera versión en el Model Registry.

In [18]:
import mlflow
import mlflow.sklearn
from functions import configure_mlflow, MLFLOW_EXPERIMENT

REGISTERED_NAME = "clasificador_riesgo_ia"

configure_mlflow()
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name="mejor_modelo_produccion") as run:
    run_id = run.info.run_id

    mlflow.log_params({
        "model_type":         seleccion["model_type"],
        "tfidf_max_features": 5000,
        "tfidf_ngram_range":  "(1, 2)",
        "tfidf_sublinear_tf": True,
        "features":           features_label,
        "experimento":        seleccion["experimento"],
    })

    mlflow.log_metrics({
        "test_f1_macro": seleccion["test_f1_macro"],
        "test_accuracy": seleccion["test_accuracy"],
        "test_roc_auc":  seleccion["test_roc_auc"],
    })

    mlflow.set_tags({
        "best_model": "true",
        "criterio":   seleccion["criterio"],
    })

    mlflow.log_artifact(path_modelo)
    mlflow.log_artifact(path_tfidf)
    mlflow.log_artifact(path_meta)

    mlflow.sklearn.log_model(
        sk_model=modelo,
        artifact_path="modelo",
        registered_model_name=REGISTERED_NAME,
    )

print(f"Run completado. Run ID: {run_id}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://34.240.189.163


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\U

Run completado. Run ID: 6b2b45e50bdb4e45a5d67323f878d1ba


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 4. Transitar a stage Production en el Model Registry

In [19]:
from functions import registrar_modelo_en_registry

model_version = registrar_modelo_en_registry(
    run_id=run_id,
    artifact_path="modelo",
    registered_name=REGISTERED_NAME,
    stage="Production",
)

c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Registered model 'clasificador_riesgo_ia' already exists. Creating a new version of this model...
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. Se

✓ 'clasificador_riesgo_ia' v6 → stage='Production'


## 5. Verificación del registro

In [20]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Listar todas las versiones del modelo
versiones = client.search_model_versions(f"name='{REGISTERED_NAME}'")
print(f"Versiones registradas de '{REGISTERED_NAME}':")
for v in versiones:
    print(f"  v{v.version} | stage={v.current_stage} | run_id={v.run_id[:8]}...")

# Verificar que hay una versión en Production
prod_versions = [v for v in versiones if v.current_stage == "Production"]
if prod_versions:
    print(f"\n✓ Modelo en Production: v{prod_versions[0].version}")
else:
    print("\n⚠ No hay versiones en Production")

c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Versiones registradas de 'clasificador_riesgo_ia':
  v6 | stage=Production | run_id=6b2b45e5...
  v5 | stage=None | run_id=6b2b45e5...
  v4 | stage=Production | run_id=ec6987f8...
  v3 | stage=None | run_id=ec6987f8...
  v2 | stage=Production | run_id=dfe7409e...
  v1 | stage=None | run_id=dfe7409e...

✓ Modelo en Production: v6


## Resumen final

| Artefacto | Ruta |
|-----------|------|
| Modelo serializado | `model/mejor_modelo.joblib` |
| Vectorizador serializado | `model/mejor_modelo_tfidf.joblib` |
| Metadata | `model/model_metadata.json` |
| MLflow Registry | `clasificador_riesgo_ia` (stage: Production) |

In [21]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
# (El logging principal ya se hizo en la celda 3 con start_run.)
# Esta celda es un resumen de verificación final.
try:
    client = MlflowClient()
    versiones = client.search_model_versions(f"name='{REGISTERED_NAME}'")
    prod = [v for v in versiones if v.current_stage == "Production"]
    print(f"✓ '{REGISTERED_NAME}' — {len(versiones)} versión(es) registrada(s)")
    if prod:
        print(f"  En Production: v{prod[0].version} (run {prod[0].run_id[:8]}...)")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '34.240.189.163'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✓ 'clasificador_riesgo_ia' — 6 versión(es) registrada(s)
  En Production: v6 (run 6b2b45e5...)
